In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install libraries
!pip install rioxarray sahi ultralytics geopandas shapely pillow

In [3]:
# Impoort libraries
import numpy as np
import rioxarray
import geopandas as gpd
from shapely.geometry import shape
from shapely.ops import unary_union
from rasterio.features import shapes
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from PIL import Image
import tempfile, os

In [4]:
# Read GeoTIFF + Reproject to UTM
def read_geotiff(tif_path):
    xds = rioxarray.open_rasterio(tif_path)

    # Band count validation
    if xds.shape[0] < 3:
        raise ValueError(f"GeoTIFF only has {xds.shape[0]} bands, minimum 3 required")

    # Auto-detect UTM zone
    center_lon = float((xds.x.min() + xds.x.max()) / 2)
    center_lat = float((xds.y.min() + xds.y.max()) / 2)
    zone = int((center_lon + 180) / 6) + 1
    epsg = 32600 + zone if center_lat >= 0 else 32700 + zone
    print(f"Auto UTM: EPSG:{epsg}")

    # Reproject to UTM
    xds_utm = xds.rio.reproject(f"EPSG:{epsg}")

    meta = {
        'transform': xds_utm.rio.transform(),
        'crs': xds_utm.rio.crs,
        'height': xds_utm.rio.height,
        'width': xds_utm.rio.width,
    }

    # Convert to uint8
    image = np.transpose(xds_utm.values[:3], (1, 2, 0))
    if image.dtype != np.uint8:
        image = ((image - image.min()) /
                 (image.max() - image.min() + 1e-8) * 255).astype(np.uint8)

    # Save tmp jpg
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as f:
        tmp_path = f.name
    Image.fromarray(image).save(tmp_path, quality=95)

    return tmp_path, meta

In [5]:
# 2.Inference with SAHI
def run_sahi(img_path, model_path, conf=0.25):
    model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=model_path,
        confidence_threshold=conf,
        device="cuda",  # replace "cpu" if there is no GPU
    )

    result = get_sliced_prediction(
        img_path, model,
        slice_height=640, slice_width=640,
        overlap_height_ratio=0.2, overlap_width_ratio=0.2,
        perform_standard_pred=False,
        postprocess_type="NMM",
        postprocess_match_threshold=0.5,
        verbose=1,
    )

    return result

In [ ]:
# Convert to Geojson
def predictions_to_geojson(result, meta, output_path="output/predictions.geojson"):
    from shapely.geometry import box
    records = []

    for pred in result.object_prediction_list:
        bbox = pred.bbox  # BoundingBox object from SAHI
        minx, miny, maxx, maxy = bbox.minx, bbox.miny, bbox.maxx, bbox.maxy

        # Convert pixel coords → geographic coords
        transform = meta['transform']
        geo_minx, geo_maxy = transform * (minx, miny)
        geo_maxx, geo_miny = transform * (maxx, maxy)

        geom = box(geo_minx, geo_miny, geo_maxx, geo_maxy)

        records.append({
            'geometry': geom,
            'class': pred.category.name,
            'confidence': round(pred.score.value, 4)
        })

    if not records:
        print("No objects detected")
        return gpd.GeoDataFrame()

    gdf = gpd.GeoDataFrame(records, crs=meta['crs'])
    gdf['area_m2'] = gdf.geometry.area.round(2)

    output_dir = os.path.dirname(output_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    gdf.to_file(output_path, driver="GeoJSON")
    print(f"Saved: {output_path} | Total objects: {len(gdf)}")

    return gdf

In [ ]:
# Running Infrance
tmp_path, meta = read_geotiff("/content/drive/MyDrive/Linked_In_Content/yolo_detection/Citra_2.tif")
try:
    result = run_sahi(tmp_path, "/content/drive/MyDrive/Linked_In_Content/yolo_detection/running_hasil/yolo26m_PalmOil_detection/weights/best.pt", conf=0.25)
    gdf = predictions_to_geojson(result, meta, "/content/drive/MyDrive/Linked_In_Content/yolo_detection/running_infrance/predictions_paml_oil.geojson")
finally:
    os.remove(tmp_path)

# Summery
if not gdf.empty:
    print(gdf.groupby('class')['area_m2'].agg(['count', 'sum', 'mean']).round(2))

In [ ]:
# Convert to centroid point GeoJSON
def bbox_geojson_to_centroid(input_path, output_path="output/predictions_centroid.geojson"):

    gdf = gpd.read_file(input_path)

    if gdf.empty:
        print("GeoJSON kosong")
        return gpd.GeoDataFrame()

    # Calculate the centroid of each bounding box
    gdf_centroid = gdf.copy()
    gdf_centroid['geometry'] = gdf.geometry.centroid

    output_dir = os.path.dirname(output_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    gdf_centroid.to_file(output_path, driver="GeoJSON")
    print(f"Saved: {output_path} | Total points: {len(gdf_centroid)}")

    return gdf_centroid

# Running
gdf_centroid = bbox_geojson_to_centroid(
    input_path="/content/drive/MyDrive/Linked_In_Content/yolo_detection/running_infrance/predictions_paml_oil.geojson",
    output_path="/content/drive/MyDrive/Linked_In_Content/yolo_detection/running_infrance/predictions_centroid.geojson"
)

Saved: /content/drive/MyDrive/Linked_In_Content/yolo_detection/running_infrance/predictions_centroid.geojson | Total points: 3778
